# PS02 - Kaggle GPU Diffusion Inference Notebook

> **Environment:** Kaggle Notebooks (Free GPU - NVIDIA T4, 16GB VRAM)
> **Model:** `stabilityai/stable-diffusion-xl-base-1.0` (Pinned Revision: `462165984030d82259a11f4367a4eed129e94a7b`)
> **VAE:** `madebyollin/sdxl-vae-fp16-fix` (Pinned Revision: `4df413ca49b2d5c82a85cfde3c7600551bbfc5a7`)
> **Compliance:** Free accelerator only. Google Colab is strictly forbidden. No paid API keys.

In [ ]:
# Step 1: Install pinned dependencies (numpy < 2 avoids C-ABI incompatibility on Kaggle)
!pip install --quiet "numpy<2.0.0" diffusers==0.29.2 transformers==4.42.3 accelerate==0.31.0 safetensors==0.4.3 torchvision

In [ ]:
# Step 2: Verify GPU and imports
import os
import json
import time
import hashlib
from datetime import datetime, timezone
from pathlib import Path
import torch
from diffusers import StableDiffusionXLPipeline, AutoencoderKL

print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU Device: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB')

In [ ]:
# Step 3: Locate or inject Job Bundle
import os
import json
from datetime import datetime, timezone

job_path_env = os.environ.get('KAGGLE_JOB_PATH', '/kaggle/input/avatar-job/job.json')

default_job = {
    'job_id': 'kaggle-demo-run-001',
    'created_at': datetime.now(timezone.utc).isoformat(),
    'model': {
        'name': 'stable-diffusion-xl-base-1.0',
        'repo': 'stabilityai/stable-diffusion-xl-base-1.0',
        'revision': '462165984030d82259a11f4367a4eed129e94a7b',
        'vae_repo': 'madebyollin/sdxl-vae-fp16-fix',
        'vae_revision': '4df413ca49b2d5c82a85cfde3c7600551bbfc5a7'
    },
    'prompt': 'a feminine young adult person, Fitzpatrick III skin tone, shoulder-length wavy dark brown hair, wearing business casual blazer, white shirt, neutral front-facing pose, modern office, soft bokeh, professional portrait photography, sharp focus, high resolution, 8k, photorealistic, studio lighting',
    'negative_prompt': 'nsfw, explicit, nudity, violence, gore, ugly, deformed, blurry, low quality, watermark, signature, text, logo, duplicate, mutation, bad anatomy, extra limbs, cartoon, anime, illustration, painting',
    'seed': 424242,
    'inference_params': {
        'steps': 20,
        'guidance_scale': 7.5,
        'aspect_ratio': '1:1',
        'output_width': 1024
    },
    'provenance_route': 'kaggle'
}

if os.path.exists(job_path_env):
    print(f'Loading job bundle from: {job_path_env}')
    with open(job_path_env, 'r', encoding='utf-8') as f:
        job = json.load(f)
else:
    print('No external job file found; using default baseline job bundle.')
    job = default_job

print(f'Job ID: {job["job_id"]}')
print(f'Seed: {job["seed"]}')
print(f'Prompt: {job["prompt"][:80]}...')

In [ ]:
# Step 4: Load Pinned SDXL Pipeline with FP16 VAE Fix
import torch
from diffusers import StableDiffusionXLPipeline, AutoencoderKL

model_repo = job['model'].get('repo', 'stabilityai/stable-diffusion-xl-base-1.0')
model_rev = job['model'].get('revision', '462165984030d82259a11f4367a4eed129e94a7b')
vae_repo = job['model'].get('vae_repo', 'madebyollin/sdxl-vae-fp16-fix')
vae_rev = job['model'].get('vae_revision', '4df413ca49b2d5c82a85cfde3c7600551bbfc5a7')

print(f'Loading VAE: {vae_repo} ({vae_rev[:10]})')
vae = AutoencoderKL.from_pretrained(
    vae_repo,
    revision=vae_rev,
    torch_dtype=torch.float16
)

print(f'Loading SDXL: {model_repo} ({model_rev[:10]})')
pipe = StableDiffusionXLPipeline.from_pretrained(
    model_repo,
    revision=model_rev,
    vae=vae,
    torch_dtype=torch.float16,
    use_safetensors=True,
    variant='fp16'
)
pipe.enable_attention_slicing()
pipe = pipe.to('cuda')
print('Model successfully loaded on GPU.')

In [ ]:
# Step 5: Resolve dimensions and run deterministic inference
import time
import hashlib
from pathlib import Path
import torch

aspect = job['inference_params'].get('aspect_ratio', '1:1')
dim_map = {
    '1:1': (1024, 1024),
    '3:4': (896, 1152),
    '9:16': (768, 1344)
}
width, height = dim_map.get(aspect, (1024, 1024))
steps = job['inference_params'].get('steps', 20)
guidance = job['inference_params'].get('guidance_scale', 7.5)
seed = job['seed']

generator = torch.Generator(device='cuda').manual_seed(seed)

output_dir = Path('/kaggle/working/output')
output_dir.mkdir(parents=True, exist_ok=True)

print(f'Generating avatar: {width}x{height}, {steps} steps, seed={seed}...')
t_start = time.perf_counter()

image = pipe(
    prompt=job['prompt'],
    negative_prompt=job['negative_prompt'],
    width=width,
    height=height,
    num_inference_steps=steps,
    guidance_scale=guidance,
    generator=generator
).images[0]

t_elapsed = time.perf_counter() - t_start
print(f'Inference complete in {t_elapsed:.2f}s')

img_filename = f'avatar_{job["job_id"]}_{seed}.png'
img_path = output_dir / img_filename
image.save(img_path)

# Compute SHA256
h = hashlib.sha256()
with open(img_path, 'rb') as f:
    for chunk in iter(lambda: f.read(65536), b''):
        h.update(chunk)
img_hash = h.hexdigest()

# Save result_fragment.json for avatarpipe ingest
fragment = {
    'job_id': job['job_id'],
    'images': [str(img_path)],
    'seed': seed,
    'model_id': job['model']['name'],
    'revision': job['model']['revision'],
    'provenance_route': 'kaggle',
    'generated_at': datetime.now(timezone.utc).isoformat(),
    'inference_time_seconds': round(t_elapsed, 3),
    'gpu_device': torch.cuda.get_device_name(0),
    'image_hashes': {
        img_filename: img_hash
    }
}

fragment_file = output_dir / 'result_fragment.json'
with open(fragment_file, 'w', encoding='utf-8') as f:
    json.dump(fragment, f, indent=2)

print(f'Saved image: {img_path}')
print(f'Saved fragment: {fragment_file}')
print('Ready for downloading and ingestion via avatarpipe ingest!')